# Projeto 4 — Pré-processamento de dados de churn

Gustavo Abreu | EBAC — Ciência de Dados | Módulo 14

Execute as células em ordem, com o CSV original na mesma pasta. Todos os resultados abaixo foram executados sobre a base fornecida.

## 1. Carregamento e diagnóstico

A base usa ponto e vírgula como separador. O identificador deve continuar textual; valores financeiros são numéricos; Idoso é categórico binário, apesar de estar codificado como 0/1.

In [1]:
import pandas as pd
from pathlib import Path
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 160)
arquivo = Path('CHURN_TELECON_MOD08_TAREFA.csv')
df = pd.read_csv(arquivo, sep=';')
print('Dimensão inicial:', df.shape)
print(df.dtypes.to_string())
print('IDs duplicados:', df['customerID'].duplicated().sum())

Dimensão inicial: (2500, 16)
customerID             object
Genero                 object
Idoso                   int64
Casado                 object
Dependents             object
Tempo_como_Cliente      int64
PhoneService           object
Servico_Internet       object
Servico_Seguranca      object
Suporte_Tecnico        object
StreamingTV            object
Tipo_Contrato          object
PaymentMethod          object
Pagamento_Mensal      float64
Total_Pago            float64
Churn                  object
IDs duplicados: 0


## 2. Correção dos tipos

Convertemos explicitamente dinheiro para float64, meses para Int64 e identificador para string. Espaços vazios são tratados como ausentes. Os tipos category serão aplicados após corrigir os rótulos.

In [2]:
texto = df.select_dtypes(include=['object', 'string']).columns
for coluna in texto:
    df[coluna] = df[coluna].astype('string').str.strip()
    df[coluna] = df[coluna].replace('', pd.NA)
for coluna in ['Pagamento_Mensal', 'Total_Pago']:
    df[coluna] = pd.to_numeric(df[coluna], errors='coerce').astype('float64')
df['Tempo_como_Cliente'] = pd.to_numeric(
    df['Tempo_como_Cliente'], errors='coerce').astype('Int64')
df['Idoso'] = df['Idoso'].astype('category')
print(df.dtypes.to_string())

customerID            string[python]
Genero                string[python]
Idoso                       category
Casado                string[python]
Dependents            string[python]
Tempo_como_Cliente             Int64
PhoneService          string[python]
Servico_Internet      string[python]
Servico_Seguranca     string[python]
Suporte_Tecnico       string[python]
StreamingTV           string[python]
Tipo_Contrato         string[python]
PaymentMethod         string[python]
Pagamento_Mensal             float64
Total_Pago                   float64
Churn                 string[python]


## 3. Percentual de ausentes

O percentual usa as 2.500 linhas originais, após normalizar espaços e converter tipos. Mostramos todas as colunas, inclusive aquelas com 0% de ausência.

In [3]:
ausentes = pd.DataFrame({
    'Quantidade': df.isna().sum(),
    'Percentual (%)': df.isna().mean().mul(100).round(2)
})
print(ausentes.to_string())

                    Quantidade  Percentual (%)
customerID                   0            0.00
Genero                      12            0.48
Idoso                        0            0.00
Casado                       0            0.00
Dependents                   0            0.00
Tempo_como_Cliente           0            0.00
PhoneService              1482           59.28
Servico_Internet             0            0.00
Servico_Seguranca            0            0.00
Suporte_Tecnico              0            0.00
StreamingTV                  0            0.00
Tipo_Contrato                0            0.00
PaymentMethod                0            0.00
Pagamento_Mensal           325           13.00
Total_Pago                   0            0.00
Churn                        5            0.20


## 4. Exclusões justificadas

PhoneService: 59,28% de ausência; a moda atribuiria serviço telefônico à maioria dos clientes sem evidência. Excluímos a coluna, aceitando a perda dessa informação. Churn: 5 ausências (0,20%); excluímos apenas essas linhas, pois a variável alvo não deve ser inventada. ID e demais colunas completas são preservados. Não removemos outliers nem duplicatas sem evidência.

In [4]:
df.drop(columns=['PhoneService'], inplace=True)
linhas_antes = len(df)
df.dropna(subset=['Churn'], inplace=True)
print('Coluna excluída: PhoneService')
print('Linhas sem Churn excluídas:', linhas_antes - len(df))
print('Dimensão após exclusões:', df.shape)
print('Ausentes restantes:')
print(df.isna().sum()[lambda serie: serie > 0].to_string())

Coluna excluída: PhoneService
Linhas sem Churn excluídas: 5
Dimensão após exclusões: (2495, 15)
Ausentes restantes:
Genero                7
Pagamento_Mensal    320


## 5. Padronização de categorias

Antes da moda, unificamos M/male em Male e F/f/female em Female. DSL/dsl são o mesmo serviço. Não confundimos No internet service com No: a ausência de internet é uma condição distinta. Mantemos os nomes originais para facilitar a comparação com o enunciado.

In [5]:
print('Gênero antes:', df['Genero'].value_counts().to_dict())
print('Internet antes:', df['Servico_Internet'].value_counts().to_dict())
df['Genero'] = df['Genero'].str.casefold().replace({
    'm': 'Male', 'male': 'Male', 'f': 'Female', 'female': 'Female'})
df['Servico_Internet'] = df['Servico_Internet'].str.casefold().replace({
    'dsl': 'DSL', 'fiber optic': 'Fiber optic', 'no': 'No'})
print('Gênero depois:', df['Genero'].value_counts().to_dict())
print('Internet depois:', df['Servico_Internet'].value_counts().to_dict())

Gênero antes: {'Male': 1265, 'Female': 1212, 'M': 4, 'f': 4, 'F': 3}
Internet antes: {'Fiber optic': 1124, 'DSL': 835, 'No': 528, 'dsl': 8}
Gênero depois: {'Male': 1269, 'Female': 1219}
Internet depois: {'Fiber optic': 1124, 'DSL': 843, 'No': 528}


## 6. Distribuição e escolha da imputação

A moda é calculada após padronizar gênero. Para Pagamento_Mensal, comparamos média, mediana, quartis e assimetria. A mediana é uma escolha robusta; a diferença entre média e mediana e a heterogeneidade dos planos justificam essa opção, sem afirmar que há assimetria extrema. A imputação global reduz variabilidade e não recupera o valor real de cada cliente.

In [6]:
print('Distribuição de Pagamento_Mensal:')
print(df['Pagamento_Mensal'].describe().round(2).to_string())
print('Assimetria:', round(df['Pagamento_Mensal'].skew(), 3))
print('Frequências de gênero:')
print(df['Genero'].value_counts().to_string())
mediana = df['Pagamento_Mensal'].median()
moda = df['Genero'].mode().iloc[0]
print('Mediana escolhida:', mediana)
print('Moda escolhida:', moda)

Distribuição de Pagamento_Mensal:
count    2175.00
mean       65.61
std        29.93
min        18.40
25%        39.50
50%        71.45
75%        90.25
max       118.65
Assimetria: -0.28
Frequências de gênero:
Genero
Male      1269
Female    1219
Mediana escolhida: 71.45
Moda escolhida: Male


## 7. Substituição dos valores ausentes

Usamos fillna com inplace=True no DataFrame. Isso atende ao requisito e evita preencher uma Series temporária, operação que pode não atualizar o DataFrame em versões com Copy-on-Write. Em futura modelagem, estimar mediana/moda apenas no conjunto de treino para evitar vazamento de dados.

In [7]:
preenchidos = df[['Genero', 'Pagamento_Mensal']].isna().sum()
df.fillna({'Genero': moda, 'Pagamento_Mensal': mediana}, inplace=True)
print('Quantidade de valores preenchidos:')
print(preenchidos.to_string())
print('Total de ausentes após imputação:', int(df.isna().sum().sum()))
for coluna in df.select_dtypes(include=['object', 'string']).columns:
    if coluna != 'customerID':
        df[coluna] = df[coluna].astype('category')
print('Tipos finais:')
print(df.dtypes.to_string())

Quantidade de valores preenchidos:
Genero                7
Pagamento_Mensal    320
Total de ausentes após imputação: 0
Tipos finais:
customerID            string[python]
Genero                      category
Idoso                       category
Casado                      category
Dependents                  category
Tempo_como_Cliente             Int64
Servico_Internet            category
Servico_Seguranca           category
Suporte_Tecnico             category
StreamingTV                 category
Tipo_Contrato               category
PaymentMethod               category
Pagamento_Mensal             float64
Total_Pago                   float64
Churn                       category


## 8. Validação e exportação

Validamos os rótulos esperados, ausência de nulos e unicidade dos IDs. O CSV não armazena metadados de tipos: ao reabrir, reaplique as conversões do notebook. Entrega final: 2.495 clientes, 15 colunas. A base foi preparada, sem treinar um modelo.

In [8]:
esperados = {
    'Genero': {'Male', 'Female'}, 'Idoso': {0, 1},
    'Casado': {'Yes', 'No'}, 'Dependents': {'Yes', 'No'},
    'Servico_Internet': {'DSL', 'Fiber optic', 'No'},
    'Servico_Seguranca': {'Yes', 'No', 'No internet service'},
    'Suporte_Tecnico': {'Yes', 'No', 'No internet service'},
    'StreamingTV': {'Yes', 'No', 'No internet service'},
    'Tipo_Contrato': {'Month-to-month', 'One year', 'Two year'},
    'PaymentMethod': {'Electronic check', 'Mailed check',
                      'Bank transfer (automatic)', 'Credit card (automatic)'},
    'Churn': {'Yes', 'No'}
}
for coluna, valores in esperados.items():
    assert set(df[coluna].dropna()).issubset(valores), coluna
assert not df.isna().any().any()
assert df['customerID'].is_unique
assert df.shape == (2495, 15)
assert df['Tempo_como_Cliente'].ge(0).all()
assert df[['Pagamento_Mensal', 'Total_Pago']].ge(0).all().all()
df.to_csv('Projeto4_Base_Limpa.csv', sep=';', index=False)
print('Validações aprovadas.')
print('Dimensão final:', df.shape)
print('Clientes por churn:', df['Churn'].value_counts().to_dict())
print('Arquivo exportado: Projeto4_Base_Limpa.csv')

Validações aprovadas.
Dimensão final: (2495, 15)
Clientes por churn: {'No': 1846, 'Yes': 649}
Arquivo exportado: Projeto4_Base_Limpa.csv
